# Narrative prompt playground

Iterate on the system prompt (and model/token settings) used by `--narrate-best`
(see `src/weather_ensemble/narrative.py`) before touching production code.

This notebook **never writes to `best_narratives`** - it only calls the Claude API
directly to preview candidate narratives, so there's nothing to clean up afterwards
no matter how many times you re-run a cell. It reuses the real
`_fetch_best_prediction` / `_fetch_best_periods` / `_build_user_prompt` functions from
`narrative.py`, so the *data* going into the prompt is always exactly what production
sees - only the prompt text/model/params below are yours to edit.

Workflow: edit `SYSTEM_PROMPT` (or `MODEL`/`MAX_OUTPUT_TOKENS`) in the cell below, re-run
the batch-test cell, read the output across a spread of real weather conditions, repeat.
Once you're happy, copy the final values back into `narrative.py`'s module-level
constants - this notebook doesn't do that for you.

In [1]:
import sqlite3
from datetime import date
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv(Path("../.env") if Path("../.env").exists() else Path(".env"))

import sys
sys.path.insert(0, str(Path("../src").resolve()))
from weather_ensemble import narrative
from weather_ensemble.config import AUSTRALIAN_LOCATIONS

DB_PATH = Path("../data/weather.db") if Path("../data/weather.db").exists() else Path("data/weather.db")
LOCATIONS_BY_NAME = {loc.name: loc for loc in AUSTRALIAN_LOCATIONS}
client = anthropic.Anthropic()

## The prompt - edit these to iterate

Starts as a copy of `narrative.py`'s current defaults. Change freely; nothing here
touches the source file until you copy it back yourself.

In [2]:
SYSTEM_PROMPT = narrative.SYSTEM_PROMPT
MODEL = narrative.MODEL
MAX_OUTPUT_TOKENS = narrative.MAX_OUTPUT_TOKENS

# $ per million tokens - Haiku 4.5 pricing as of this notebook's writing. Update
# if you switch MODEL to compare against Sonnet/Opus, or if pricing changes.
PRICE_PER_MTOK_IN = 1.00
PRICE_PER_MTOK_OUT = 5.00

print(SYSTEM_PROMPT)


    You are a weather reporter. You will write a single brief weather summary from the 
    forecast data given. Cover the max and min temperature, whether it will rain and 
    roughly when in the day if so, whether it will be mostly sunny or cloudy, and whether
    its will be windy. The target audience is a regular person who wants a brief summary 
    of tomorrow's weather. Do not elaborate about what to wear or securing loose items.
    Be consistent when describing the weather, and do not contradict yourself. 
    
    Plain prose only - no headings, bullet points, or markdown formatting. Do not mention 
    data sources, models, probabilities as percentages, or units you were not given.
    Round to the nearest whole number when referencing units.
    


## Pick a spread of test locations

Pulls the latest forecast date and a handful of locations spanning dry/rainy/hot/cold/
windy conditions, straight from the real database - so the spread stays meaningful
even as the data moves forward day to day, rather than pointing at hardcoded dates
that go stale.

In [3]:
conn = sqlite3.connect(DB_PATH)
latest_date = conn.execute("SELECT MAX(forecast_date) FROM best_predictions").fetchone()[0]
TARGET_DATE = date.fromisoformat(latest_date)

rows = conn.execute(
    "SELECT location_name, precipitation_sum, wind_speed, max_temp FROM best_predictions "
    "WHERE forecast_date = ? ORDER BY precipitation_sum DESC",
    (latest_date,),
).fetchall()

# Evenly spaced across the precipitation-sorted list: covers the wettest, the
# driest, and a few points in between in one small, repeatable sample.
n = len(rows)
sample_idx = sorted({0, n // 4, n // 2, 3 * n // 4, n - 1})
TEST_LOCATIONS = [rows[i][0] for i in sample_idx]

print(f"Latest forecast date: {TARGET_DATE}")
print("Test spread:")
for i in sample_idx:
    name, precip, wind, temp = rows[i]
    print(f"  {name:<20} precip={precip:>5.1f}mm  wind={wind:>5.1f}km/h  max_temp={temp:>5.1f}C")

Latest forecast date: 2026-08-03
Test spread:
  Margaret River       precip= 11.1mm  wind= 25.8km/h  max_temp= 18.0C
  Sydney               precip=  2.1mm  wind= 13.8km/h  max_temp= 20.9C
  Lorne                precip=  0.8mm  wind= 24.1km/h  max_temp= 12.7C
  Darwin               precip=  0.0mm  wind= 19.4km/h  max_temp= 30.6C
  Kangaroo Island      precip=  0.0mm  wind= 19.2km/h  max_temp= 14.0C


## Generate + preview one location

`preview()` builds the prompt from real Best-prediction data via `narrative.py`'s own
functions, calls the API with the editable settings above, and prints everything -
the input prompt, the narrative, and token usage/cost for that one call.

In [4]:
def preview(location_name: str, target_date: date = None, verbose: bool = True):
    target_date = target_date or TARGET_DATE
    location = LOCATIONS_BY_NAME[location_name]
    prediction = narrative._fetch_best_prediction(DB_PATH, location, target_date)
    if prediction is None:
        print(f"No Best prediction for {location_name} on {target_date} - try a different date/location.")
        return None
    periods = narrative._fetch_best_periods(DB_PATH, location, target_date)
    prompt = narrative._build_user_prompt(location, target_date, prediction, periods)

    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_OUTPUT_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    text = next((b.text for b in response.content if b.type == "text"), None)
    cost = response.usage.input_tokens / 1e6 * PRICE_PER_MTOK_IN + response.usage.output_tokens / 1e6 * PRICE_PER_MTOK_OUT

    if verbose:
        print("=" * 70)
        print(f"{location_name}  ({target_date})")
        print("-" * 70)
        print("PROMPT:")
        print(prompt)
        print("-" * 70)
        print("NARRATIVE:")
        print(text)
        print()
        print(f"tokens: in={response.usage.input_tokens} out={response.usage.output_tokens}  cost=${cost:.6f}")

    return {"location": location_name, "prompt": prompt, "narrative": text, "usage": response.usage, "cost": cost}


_ = preview(TEST_LOCATIONS[0])

Margaret River  (2026-08-03)
----------------------------------------------------------------------
PROMPT:
Location: Margaret River
Date: 2026-08-03
Max/min temperature: 18.0C / 7.5C
Chance of rain: 100%
Total precipitation: 11.1mm
Precipitation by period: overnight 0.0mm, morning 0.0mm, afternoon 0.0mm, evening 11.1mm
Cloud cover: 61%
Wind: 26 km/h (gusts 55 km/h)
----------------------------------------------------------------------
NARRATIVE:
Tomorrow in Margaret River will be a cool winter's day with a high of 18 degrees and a low of 8 degrees. Rain is expected in the evening, bringing around 11 millimetres of precipitation. It will be mostly cloudy throughout the day with moderate to strong winds, including gusts up to 55 kilometres per hour.

tokens: in=290 out=76  cost=$0.000670


## Batch test across the diverse spread

Runs the same prompt/model settings across every location in `TEST_LOCATIONS` in one
go, so you can eyeball consistency across very different weather conditions at once,
plus the total/projected nightly cost for all 30 locations.

In [5]:
results = [preview(name) for name in TEST_LOCATIONS]
results = [r for r in results if r is not None]

total_cost = sum(r["cost"] for r in results)
print("=" * 70)
print(f"TOTAL across {len(results)} locations: ${total_cost:.6f}")
print(f"Projected nightly cost for all 30 locations: ${total_cost / len(results) * 30:.6f}")

Margaret River  (2026-08-03)
----------------------------------------------------------------------
PROMPT:
Location: Margaret River
Date: 2026-08-03
Max/min temperature: 18.0C / 7.5C
Chance of rain: 100%
Total precipitation: 11.1mm
Precipitation by period: overnight 0.0mm, morning 0.0mm, afternoon 0.0mm, evening 11.1mm
Cloud cover: 61%
Wind: 26 km/h (gusts 55 km/h)
----------------------------------------------------------------------
NARRATIVE:
Tomorrow in Margaret River will be cool with a high of 18 degrees and a low of 8 degrees. Rain is expected in the evening, bringing about 11 millimetres of precipitation. It will be mostly cloudy throughout the day with moderate to strong winds, gusting up to 55 kilometres per hour.

tokens: in=290 out=71  cost=$0.000645
Sydney  (2026-08-03)
----------------------------------------------------------------------
PROMPT:
Location: Sydney
Date: 2026-08-03
Max/min temperature: 20.9C / 8.4C
Chance of rain: 39%
Total precipitation: 2.1mm
Precipitati

## When you're happy with the prompt

Copy the final `SYSTEM_PROMPT` (and `MODEL` / `MAX_OUTPUT_TOKENS` if you changed them)
back into the matching constants near the top of `src/weather_ensemble/narrative.py`.
Nothing in this notebook writes to the database, so there's no cleanup needed either
way - only editing `narrative.py` makes a change stick.